# RAG Pipeline Evaluation — Energy Documents

**Stack:** LangChain LCEL | FAISS | Groq llama-3.1-8b-instant | HuggingFace all-MiniLM-L6-v2  
**Scoring:** Embedding Cosine Similarity + LLM-as-Judge  
**LangSmith:** Integrated dataset + evaluation logging  
**Golden QA Set:** 24 questions across 6 energy documents

In [ ]:
import sys, os, time, re, json
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

try:
    PROJECT_ROOT = Path(__file__).resolve().parents[2]
except (NameError, AttributeError):
    cwd = Path.cwd()
    PROJECT_ROOT = cwd if (cwd / "energy_docs_chat").exists() else cwd.parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

from energy_docs_chat.src.chat_with_doc.retrieval import RetrievalPipeline
from energy_docs_chat.utils.model_loader import get_llm
from energy_docs_chat.logger.custom_logger import logger

print(f"Project root: {PROJECT_ROOT}")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
GOLDEN_CSV_PATH = PROJECT_ROOT / "data" / "golden_eval.csv"
RESULTS_CSV_PATH = PROJECT_ROOT / "data" / "eval_results.csv"

EMBED_SIM_PASS_THRESHOLD = 0.55
LLM_JUDGE_PASS_THRESHOLD = 3.0
SESSION_PREFIX = "eval_isolated"
INTER_CALL_DELAY_SECONDS = 2.5
JUDGE_CALL_DELAY_SECONDS = 2.0

print("Config loaded:")
print(f"  Golden CSV: {GOLDEN_CSV_PATH}")
print(f"  Results CSV: {RESULTS_CSV_PATH}")
print(f"  Embed threshold: {EMBED_SIM_PASS_THRESHOLD}")
print(f"  Judge threshold: {LLM_JUDGE_PASS_THRESHOLD} / 5")

In [ ]:
df_golden = pd.read_csv(GOLDEN_CSV_PATH)
required_cols = {"question", "expected_answer", "document_source", "topic", "difficulty"}
assert required_cols.issubset(df_golden.columns)

print(f"Loaded {len(df_golden)} golden rows.\nDifficulty:")
print(df_golden['difficulty'].value_counts())
print(f"\nTopic:")
print(df_golden['topic'].value_counts())

In [ ]:
# LangSmith: Create & Upload Golden Dataset
from langsmith import Client

ls_client = Client()
dataset_name = "energy-docs-rag-golden-eval"

try:
    dataset = ls_client.read_dataset(dataset_name=dataset_name)
    print(f"Dataset '{dataset_name}' exists (ID: {dataset.id})")
except:
    print(f"Creating dataset '{dataset_name}'...")
    dataset = ls_client.create_dataset(
        dataset_name=dataset_name,
        description="Golden evaluation set for Energy Docs RAG: 24 Q&A pairs across 6 PDFs"
    )
    print(f"Dataset created (ID: {dataset.id})")

print(f"\nDataset ID: {dataset.id}")
print(f"View at: https://smith.langchain.com/datasets/{dataset.id}")

# Upload examples
existing = sum(1 for _ in ls_client.list_examples(dataset_id=dataset.id))
if existing < len(df_golden):
    print(f"\nUploading {len(df_golden)} examples...")
    for _, row in df_golden.iterrows():
        ls_client.create_example(
            dataset_id=dataset.id,
            inputs={
                "question": row["question"],
                "document_source": row["document_source"],
                "topic": row["topic"],
                "difficulty": row["difficulty"],
            },
            outputs={"expected_answer": row["expected_answer"]},
            metadata={"topic": row["topic"], "difficulty": row["difficulty"]},
        )
    print(f"Uploaded {len(df_golden)} examples")
else:
    print(f"Dataset already has {existing} examples")

In [ ]:
print("Initializing RAG pipeline...")
pipeline = RetrievalPipeline()
rag_chain = pipeline.build_chain()
print("RAG chain ready.")

print("Loading SentenceTransformer...")
score_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Scorer ready.")

print("Initializing judge LLM...")
judge_llm = get_llm()
print("Judge LLM ready.")

In [ ]:
def compute_embedding_similarity(expected: str, actual: str, model) -> float:
    if not actual or len(actual.strip()) < 5:
        return 0.0
    vecs = model.encode([expected, actual], normalize_embeddings=True)
    sim = float(np.dot(vecs[0], vecs[1]))
    return round(max(0.0, sim), 4)

test_sim = compute_embedding_similarity(
    "The frequency range is 50 Hz +/- 200 mHz.",
    "Standard frequency must stay between 49.8 Hz and 50.2 Hz.",
    score_model
)
print(f"Embedding sanity check: {test_sim:.4f} (should be > 0.7)")

In [ ]:
LLM_JUDGE_PROMPT = """You are a judge for a RAG system answering energy questions.

Score ACTUAL ANSWER vs EXPECTED ANSWER on:
- Faithfulness (1-3): No hallucinations? (3=faithful, 2=minor error, 1=hallucinated)
- Relevance (1-2): Addresses question? (2=direct, 1=partial)

TOTAL = Faithfulness + Relevance (2-5 range)

Respond ONLY with JSON:
{"score": <2-5>, "faithfulness": <1-3>, "relevance": <1-2>}

QUESTION: {question}
EXPECTED: {expected_answer}
ACTUAL: {actual_answer}
"""

def compute_llm_judge_score(question: str, expected: str, actual: str, llm) -> dict:
    if not actual or len(actual.strip()) < 5:
        return {"score": 1, "faithfulness": 1, "relevance": 1}
    
    prompt = LLM_JUDGE_PROMPT.format(question=question, expected_answer=expected, actual_answer=actual)
    try:
        response = llm.invoke(prompt)
        json_match = re.search(r'\{.*?\}', response.content.strip(), re.DOTALL)
        if json_match:
            result = json.loads(json_match.group())
            result["score"] = max(2, min(5, int(result.get("score", 2))))
            return result
    except:
        pass
    return {"score": 2, "faithfulness": 1, "relevance": 1}

test_judge = compute_llm_judge_score(
    "What is the Standard Frequency Range?",
    "±200 mHz for CE",
    "The standard frequency range for Continental Europe is ±200 millihertz.",
    judge_llm
)
print(f"Judge sanity check: {test_judge}")

In [ ]:
results = []
print(f"Evaluating {len(df_golden)} questions...\n" + "=" * 70)

for i, row in df_golden.iterrows():
    try:
        actual = rag_chain.invoke(
            {"question": row["question"]},
            config={"configurable": {"session_id": f"eval_{i}"}}
        )
    except Exception as e:
        actual = ""
    
    time.sleep(INTER_CALL_DELAY_SECONDS)
    embed_sim = compute_embedding_similarity(row["expected_answer"], actual, score_model)
    judge_result = compute_llm_judge_score(row["question"], row["expected_answer"], actual, judge_llm)
    time.sleep(JUDGE_CALL_DELAY_SECONDS)
    
    results.append({
        "question": row["question"],
        "document_source": row["document_source"],
        "topic": row["topic"],
        "difficulty": row["difficulty"],
        "expected_answer": row["expected_answer"],
        "actual_answer": actual,
        "embed_sim_score": embed_sim,
        "llm_judge_score": judge_result["score"],
        "llm_faithfulness": judge_result.get("faithfulness", 1),
        "llm_relevance": judge_result.get("relevance", 1),
        "embed_pass": embed_sim >= EMBED_SIM_PASS_THRESHOLD,
        "judge_pass": judge_result["score"] >= LLM_JUDGE_PASS_THRESHOLD,
    })
    
    status = "PASS" if results[-1]["embed_pass"] and results[-1]["judge_pass"] else "FAIL"
    if (i + 1) % 6 == 0 or i == len(df_golden) - 1:
        print(f"[{i+1:02d}/{len(df_golden)}] {status}")

print("=" * 70)

In [ ]:
df_results = pd.DataFrame(results)
df_results.to_csv(RESULTS_CSV_PATH, index=False, encoding="utf-8")
print(f"Results saved: {RESULTS_CSV_PATH}")
print(f"\nSample:")
df_results[["question", "embed_sim_score", "llm_judge_score", "embed_pass", "judge_pass"]].head()

In [ ]:
total = len(df_results)
n_both = (df_results["embed_pass"] & df_results["judge_pass"]).sum()

print("\n" + "=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)
print(f"Total questions:           {total}")
print(f"Mean embed similarity:     {df_results['embed_sim_score'].mean():.4f} (threshold: {EMBED_SIM_PASS_THRESHOLD})")
print(f"Mean LLM judge score:      {df_results['llm_judge_score'].mean():.4f} (threshold: {LLM_JUDGE_PASS_THRESHOLD})")
print(f"Both metrics PASS rate:    {n_both}/{total}  ({100*n_both/total:.1f}%)")
print("=" * 60)

print("\nScores by difficulty:")
print(df_results.groupby("difficulty")[["embed_sim_score", "llm_judge_score"]].mean().round(4))
print("\nScores by topic:")
print(df_results.groupby("topic")[["embed_sim_score", "llm_judge_score"]].mean().round(4))

In [ ]:
# LangSmith: Log Evaluation Results with Scores
print(f"\nLogging {len(df_results)} runs to LangSmith...\n" + "=" * 70)

experiment = f"energy-rag-eval-{datetime.now().strftime('%Y%m%d_%H%M%S')}"
import uuid

for idx, row in df_results.iterrows():
    run_id = uuid.uuid4()
    ls_client.create_run(
        id=run_id,
        name="energy-rag-evaluation",
        run_type="chain",
        inputs={
            "question": row["question"],
            "document_source": row["document_source"],
            "topic": row["topic"],
            "difficulty": row["difficulty"],
        },
        outputs={
            "actual_answer": row["actual_answer"],
            "embed_sim_score": float(row["embed_sim_score"]),
            "llm_judge_score": int(row["llm_judge_score"]),
        },
        tags=["rag-evaluation", row["topic"], row["difficulty"]],
        metadata={
            "experiment": experiment,
            "topic": row["topic"],
            "difficulty": row["difficulty"],
            "embed_pass": bool(row["embed_pass"]),
            "judge_pass": bool(row["judge_pass"]),
        }
    )
    
    ls_client.create_feedback(
        run_id=run_id,
        key="embed_similarity",
        score=float(row["embed_sim_score"]),
    )
    ls_client.create_feedback(
        run_id=run_id,
        key="llm_judge_score",
        score=float(row["llm_judge_score"]) / 5.0,
    )
    ls_client.create_feedback(
        run_id=run_id,
        key="both_metrics_pass",
        score=1.0 if (row["embed_pass"] and row["judge_pass"]) else 0.0,
    )
    
    if (idx + 1) % 8 == 0 or idx == len(df_results) - 1:
        print(f"[{idx+1:02d}/{len(df_results)}] Logged to LangSmith")

print("=" * 70)
print(f"All evaluations logged (experiment: {experiment})")
print(f"\nView runs in LangSmith: https://smith.langchain.com/")
print(f"Filter by tags: rag-evaluation, topic, difficulty")

## LangSmith Integration Complete

**What was logged:**
- Dataset `energy-docs-rag-golden-eval` with 24 golden Q&A examples
- 24 evaluation runs tagged with `rag-evaluation`, topic, and difficulty
- 3 feedback scores per run:
  - `embed_similarity`: cosine similarity (0-1)
  - `llm_judge_score`: judge score normalized to 0-1
  - `both_metrics_pass`: 1 if both metrics passed, 0 otherwise

**Where to find results:**
1. [LangSmith Datasets](https://smith.langchain.com/) - view golden dataset
2. [LangSmith Runs](https://smith.langchain.com/) - search for runs tagged `rag-evaluation`
3. Filter by topic or difficulty to analyze performance by category

**Metrics to watch:**
- Mean embed_similarity ≥ 0.55
- Mean llm_judge_score ≥ 0.6 (normalized from 3/5)
- both_metrics_pass ≥ 0.7